# T3.1 — Ortam kurulumu + GPU testi

**Hedef ortam:** Colab Pro (A100/T4) / Kaggle GPU / yerel CUDA  
**Süre:** 0–2h  
**Çıktı:** GPU erişimi doğrulandı, segmentation_models_pytorch + timm import çalışıyor.

## Kontrol listesi
1. CUDA versiyonu + GPU bilgisi
2. PyTorch CUDA mevcut mu
3. segmentation_models_pytorch (U-Net) import
4. timm (SSL4EO-S12 backbone) import
5. rasterio + numpy + scipy import
6. grad-cam import
7. Tek tile (17, 256, 256) tensor → forward pass smoke test

## Sanity threshold
- `torch.cuda.is_available()` → True
- Forward pass output shape → `(1, 1, 256, 256)`
- VRAM kullanımı < 4 GB (256x256 batch=1 için)

In [ ]:
# 1) Bağımlılık kurulumu (Colab/Kaggle)
!pip install -q segmentation_models_pytorch==0.3.3 timm==0.9.12 grad-cam==1.5.0 \
    rasterio==1.3.9 huggingface-hub==0.20.0 albumentations==1.3.1

In [ ]:
# 2) CUDA + GPU kontrolü
import torch, sys, platform
print('Python   :', sys.version.split()[0])
print('Platform :', platform.platform())
print('Torch    :', torch.__version__)
print('CUDA av. :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA ver :', torch.version.cuda)
    print('GPU      :', torch.cuda.get_device_name(0))
    print('VRAM tot :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
    print('Capab.   :', torch.cuda.get_device_capability(0))
else:
    print('UYARI: CUDA yok — CPU fallback. Fine-tune koşmaz, smoke test sınırlı.')

In [ ]:
# 3) Library imports
import segmentation_models_pytorch as smp
import timm
import rasterio, numpy as np, scipy, sklearn
from pytorch_grad_cam import GradCAM
print('smp      :', smp.__version__)
print('timm     :', timm.__version__)
print('rasterio :', rasterio.__version__)
print('numpy    :', np.__version__)
print('scipy    :', scipy.__version__)
print('sklearn  :', sklearn.__version__)

In [ ]:
# 4) U-Net (17 ch input → 1 ch output) smoke test
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights=None,    # SSL4EO-S12 manuel yüklenecek (T3.2)
    in_channels=17,           # 13 S2 + S1 VV/VH + DEM + slope
    classes=1,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Param sayisi : {n_params/1e6:.2f} M')

x = torch.randn(1, 17, 256, 256, device=device)
with torch.no_grad():
    y = model(x)
print('Input  :', tuple(x.shape))
print('Output :', tuple(y.shape))
assert y.shape == (1, 1, 256, 256), 'Output shape uyumsuz!'
print('OK — forward pass calisti.')

In [ ]:
# 5) VRAM ölçümü (batch=4 256x256, fine-tune profili)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    x = torch.randn(4, 17, 256, 256, device=device, requires_grad=False)
    y = model(x)
    loss = y.mean()
    loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f'Peak VRAM (batch=4 fwd+bwd): {peak:.2f} GB')
    del x, y, loss
    torch.cuda.empty_cache()

## DELIVER
```
[P3] T3.1 TAMAM
Cikti: GPU erisimi + library importlar dogrulandi.
Metric: torch.cuda.is_available()=True, U-Net (17->1) forward OK.
Siradaki: T3.2 SSL4EO-S12 pretrained yukleme + 13->17 adapter.
```